# 10. Multivariate Forecasting: Cross-Variable Dependencies

## Background

Most real-world forecasting problems involve multiple correlated time series: store sales across categories, energy demand across regions, financial returns across assets. The key question: does modeling cross-variable dependencies help or hurt?

Surprisingly, the answer from recent research is often 'it depends.' PatchTST and DLinear showed that **channel-independent** models (treating each series independently) often outperform channel-mixing models on standard benchmarks. Explanations:

1. **Spurious correlations**: with many channels, models learn spurious cross-channel    patterns that don't generalize
2. **Distribution shift**: channel correlations change over time; training on historical    correlations may mislead

However, **iTransformer** (Liu et al., 2024) inverted the attention pattern — applying attention across channels (not time steps) — and achieved strong multivariate results by capturing genuine inter-series dependencies.

## What You'll Learn

- Multivariate dataset creation: correlated synthetic series
- Channel-independent vs channel-mixing baselines
- iTransformer: inverted attention for cross-series modeling
- When to use multivariate vs univariate forecasting

## Why This Matters

Supply chain planning, energy grid management, and financial risk all require joint forecasts of correlated series. Understanding when channel mixing helps — and the failure modes when it hurts — prevents the common mistake of assuming multivariate = more information = better forecasts.


## Correlated Multivariate Time Series

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

np.random.seed(42)

def generate_multivariate_ts(n: int = 500, n_channels: int = 5,
                               correlation: float = 0.7) -> np.ndarray:
    '''Generate n_channels correlated time series via a factor model.
    Common factor drives shared trends; idiosyncratic noise adds per-series variation.
    '''
    t = np.arange(n)

    # Common factor: shared trend + seasonality
    common_trend = 0.02 * t
    common_seasonal = 10 * np.sin(2 * np.pi * t / 52)
    common_factor = common_trend + common_seasonal + np.random.randn(n) * 3

    series = []
    for c in range(n_channels):
        base = 50 + c * 10  # different level per channel
        loading = correlation + (1 - correlation) * np.random.rand()  # factor loading
        idiosyncratic = np.sqrt(1 - loading**2) * np.random.randn(n) * 5
        # Channel-specific phase shift
        phase = c * np.pi / n_channels
        channel_seasonal = 5 * np.sin(2 * np.pi * t / 52 + phase)
        s = base + loading * common_factor + channel_seasonal + idiosyncratic
        series.append(s)

    return np.column_stack(series)  # (n, n_channels)

mv_data = generate_multivariate_ts(n=600, n_channels=5, correlation=0.6)
print(f'Multivariate series: {mv_data.shape} (time x channels)')

# Compute correlation matrix
corr = np.corrcoef(mv_data.T)
print(f'\nCross-channel correlation matrix:')
print(np.round(corr, 2))
print(f'\nMean off-diagonal correlation: {(corr.sum() - np.trace(corr))/(corr.size - corr.shape[0]):.3f}')


## Channel-Independent vs Channel-Mixing Comparison

In [ ]:
CONTEXT, HORIZON, N_CHANNELS = 52, 12, 5
SPLIT = int(len(mv_data) * 0.8)

# Normalize
mean = mv_data[:SPLIT].mean(axis=0, keepdims=True)
std = mv_data[:SPLIT].std(axis=0, keepdims=True)
normalized = (mv_data - mean) / std

def make_mv_windows(data, context, horizon):
    X, y = [], []
    for i in range(len(data) - context - horizon):
        X.append(data[i:i+context, :])           # (context, C)
        y.append(data[i+context:i+context+horizon, :])  # (horizon, C)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = make_mv_windows(normalized, CONTEXT, HORIZON)
X_tr, y_tr = torch.tensor(X[:SPLIT-CONTEXT-HORIZON]), torch.tensor(y[:SPLIT-CONTEXT-HORIZON])
X_te, y_te = torch.tensor(X[SPLIT-CONTEXT-HORIZON:]), torch.tensor(y[SPLIT-CONTEXT-HORIZON:])

class ChannelIndependent(nn.Module):
    '''Process each channel independently with a shared MLP.'''
    def __init__(self, context: int, horizon: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(context, hidden), nn.ReLU(),
            nn.Linear(hidden, horizon)
        )
    def forward(self, x):  # x: (B, T, C)
        return self.net(x.permute(0, 2, 1)).permute(0, 2, 1)  # (B, H, C)

class ChannelMixing(nn.Module):
    '''Mix all channels with a joint MLP.'''
    def __init__(self, context: int, horizon: int, n_channels: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(context * n_channels, hidden), nn.ReLU(),
            nn.Linear(hidden, horizon * n_channels)
        )
        self.horizon = horizon
        self.n_channels = n_channels
    def forward(self, x):  # x: (B, T, C)
        B = x.shape[0]
        flat = x.reshape(B, -1)
        out = self.net(flat).reshape(B, self.horizon, self.n_channels)
        return out

def train_and_eval(model, X_tr, y_tr, X_te, y_te, epochs=30, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = nn.MSELoss()(model(X_tr), y_tr)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_te).numpy() * std + mean
        actuals = y_te.numpy() * std + mean
    mape = np.mean(np.abs((actuals - preds) / actuals)) * 100
    return mape

print('Comparing multivariate forecasting strategies:')
ci_model = ChannelIndependent(CONTEXT, HORIZON)
cm_model = ChannelMixing(CONTEXT, HORIZON, N_CHANNELS)

ci_mape = train_and_eval(ci_model, X_tr, y_tr, X_te, y_te)
cm_mape = train_and_eval(cm_model, X_tr, y_tr, X_te, y_te)

print(f'  Channel-Independent MAPE: {ci_mape:.2f}%')
print(f'  Channel-Mixing MAPE:      {cm_mape:.2f}%')
print(f'  Winner: {"Channel-Independent" if ci_mape < cm_mape else "Channel-Mixing"}')
print(f'\nNote: On long-horizon tasks, CI often wins due to distribution shift in cross-series correlations.')
